# Discovery + Silver: `university.courses`

Mismo patron. Esta tabla tiene FK a `professors` (`professor_id`), asi que ademas de limpiar tipos chequeamos integridad referencial contra `silver.university__professors` (ya cargada en la notebook anterior).

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM bronze.university__courses", engine)
df.shape

(300, 9)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

course_id               object
code                    object
name                    object
credits                 object
department              object
professor_id            object
_source_file            object
_ingested_at    datetime64[ns]
_dag_run_id             object
dtype: object


,course_id,code,name,credits,department,professor_id,_source_file,_ingested_at,_dag_run_id
0,CRS-00001,C-00001,Course 00001,3,cs,PRF-00153,university/courses.csv,2026-07-17 15:16:32.719288,manual__2026-07-17T15:16:30+00:00
1,CRS-00002,C-00002,Course 00002,2,math,PRF-00186,university/courses.csv,2026-07-17 15:16:32.719288,manual__2026-07-17T15:16:30+00:00
2,CRS-00003,C-00003,Course 00003,4,literature,PRF-00025,university/courses.csv,2026-07-17 15:16:32.719288,manual__2026-07-17T15:16:30+00:00
3,CRS-00004,C-00004,Course 00004,6,physics,PRF-00163,university/courses.csv,2026-07-17 15:16:32.719288,manual__2026-07-17T15:16:30+00:00
4,CRS-00005,C-00005,Course 00005,5,biology,PRF-00130,university/courses.csv,2026-07-17 15:16:32.719288,manual__2026-07-17T15:16:30+00:00


## 2. Nulos, duplicados e integridad referencial

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("course_id duplicados:", df["course_id"].duplicated().sum())

professors = pd.read_sql("SELECT professor_id FROM silver.university__professors", engine)
huerfanos = ~df["professor_id"].isin(professors["professor_id"])
print("course.professor_id huerfanos (sin professor valido):", huerfanos.sum())

Nulos por columna:
course_id       0
code            0
name            0
credits         0
department      0
professor_id    0
_source_file    0
_ingested_at    0
_dag_run_id     0
dtype: int64

course_id duplicados: 0
course.professor_id huerfanos (sin professor valido): 0


## 3. `department` del curso vs `department` del profesor

Hallazgo ya documentado en `docs/calidad_datos.md`: nunca coinciden. Lo confirmamos aca y decidimos **no reconciliarlos** -- son dos atributos independientes (area del curso vs area de origen del profesor asignado), no un error de join.

In [4]:
profs_dept = pd.read_sql("SELECT professor_id, department AS professor_department FROM silver.university__professors", engine)
merged = df.merge(profs_dept, on="professor_id", how="left")
coincide = merged["department"].str.strip().str.lower() == merged["professor_department"]
print("Cursos donde course.department == professor.department:", coincide.sum(), "/", len(merged))
print()
print("Valores distintos de course.department:")
print(df["department"].value_counts())

Cursos donde course.department == professor.department: 36 / 300

Valores distintos de course.department:
department
physics       52
cs            47
biology       46
economics     35
literature    34
chemistry     32
history       31
math          23
Name: count, dtype: int64


## 4. `credits`: rango de valores

In [5]:
credits = pd.to_numeric(df["credits"], errors="coerce")
print("credits no numericos:", credits.isna().sum())
print("credits <= 0:", (credits <= 0).sum())
print(credits.describe())

credits no numericos: 0
credits <= 0: 0
count    300.000000
mean       3.980000
std        1.439853
min        2.000000
25%        3.000000
50%        4.000000
75%        5.000000
max        6.000000
Name: credits, dtype: float64


## 5. Conclusion: reglas de limpieza

Tabla limpia estructuralmente (sin nulos, sin duplicados, 0 FKs huerfanas hacia `professors`). Reglas:

- `code`, `name` -> `strip()`.
- `department` -> `strip()` + minusculas (se mantiene como atributo propio del curso, **no se reconcilia** con el department del profesor).
- `credits` -> castear a entero.
- `professor_id` -> se mantiene tal cual (FK ya validada, 0 huerfanos).

## 6. Limpieza con pandas

In [6]:
df_silver = df[["course_id", "code", "name", "credits", "department", "professor_id"]].copy()

df_silver["code"] = df_silver["code"].str.strip()
df_silver["name"] = df_silver["name"].str.strip()
df_silver["department"] = df_silver["department"].str.strip().str.lower()
df_silver["credits"] = pd.to_numeric(df_silver["credits"], errors="raise").astype(int)

df_silver.head()

,course_id,code,name,credits,department,professor_id
0,CRS-00001,C-00001,Course 00001,3,cs,PRF-00153
1,CRS-00002,C-00002,Course 00002,2,math,PRF-00186
2,CRS-00003,C-00003,Course 00003,4,literature,PRF-00025
3,CRS-00004,C-00004,Course 00004,6,physics,PRF-00163
4,CRS-00005,C-00005,Course 00005,5,biology,PRF-00130


## 7. Validar antes de escribir

In [7]:
assert len(df_silver) == len(df), "se perdieron o duplicaron filas en la limpieza"
assert df_silver.isna().sum().sum() == 0, "aparecieron nulos nuevos"
assert df_silver["course_id"].is_unique, "course_id ya no es unico"
assert df_silver["professor_id"].isin(professors["professor_id"]).all(), "aparecio una FK huerfana"
print("OK:", len(df_silver), "filas listas para silver")

OK: 300 filas listas para silver


## 8. Escribir en `silver.university__courses`

In [8]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

df_silver.to_sql(
    "university__courses",
    engine,
    schema="silver",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=1000,
)
print("Escrito en silver.university__courses")

Escrito en silver.university__courses


## 9. Verificar lo que quedo en Postgres

In [9]:
check = pd.read_sql("SELECT * FROM silver.university__courses LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(DISTINCT course_id) AS ids_unicos FROM silver.university__courses", engine))
check

   filas  ids_unicos
0    300         300


,course_id,code,name,credits,department,professor_id,_silver_loaded_at
0,CRS-00001,C-00001,Course 00001,3,cs,PRF-00153,2026-07-17 15:17:10.123215+00:00
1,CRS-00002,C-00002,Course 00002,2,math,PRF-00186,2026-07-17 15:17:10.123215+00:00
2,CRS-00003,C-00003,Course 00003,4,literature,PRF-00025,2026-07-17 15:17:10.123215+00:00
3,CRS-00004,C-00004,Course 00004,6,physics,PRF-00163,2026-07-17 15:17:10.123215+00:00
4,CRS-00005,C-00005,Course 00005,5,biology,PRF-00130,2026-07-17 15:17:10.123215+00:00
